# Notebook 01 — Data Loading and Merging

## Overview

This first notebook builds the project dataset for Q1 from 4 separate public sources. Since here our business objective is to forecast the final funding tier of a specific disaster event at the time of declaration (before any final cost consideration can be made with certainity), we chose to structure the data so that each federally declared disaster represents a single row.

The merging process starts with loading the raw FEMA project-level grant data and joining it with declaration administrative data to extract the actual incident dates. Next, we constructed the FIPS code for each record to use as our keys. FIPS stands for Federal Information Processing Standards, and are unique number identifiers used by the US government to track geographic areas consistently across different databases. This allows us to merge together all the Census socioeconomic indicators with the FEMA National Risk Index. Our rationale behind this choice is that administrative hazard data alone does not fully capture a community's financial vulnerability or capacity to recover. Finally, all the county-level data is aggregated up to the disaster level.

In [1]:
import pandas as pd
import numpy as np
import sys

#add parent dir so utils.py is importable
sys.path.append('../')
from utils import data_summary

#path constants -> reused across all cells
RAW       = '../data/raw/'
PROCESSED = '../data/processed/'

## 1.1 Load FEMA Public Assistance Funded Projects

The FEMA Public Assistance (PA) dataset is sourced from OpenFEMA, the agency's open data portal. Here is the link to access it:
- https://www.fema.gov/openfema-data-page/public-assistance-funded-projects-details-v2

We chose it as it contains one row per grant project approved under the PA program, which funds repair and recovery of public infrastructure which follow federally declared disasters. More specifically, each row represents a single project work order submitted by a state or local government applicant. The key financial column for our project is `federalShareObligated`, which records the final federal dollar commitment for that project. This is the figure that we will sum to form the disaster-level total that we're using as the prediction target.

At this stage we simply load the dataset without applying any filtering or cleaning, which will come later.

In [2]:
#loading all project rows -> no filters yet
pa = pd.read_csv(RAW + 'pa_funded_projects.csv', low_memory=False)
data_summary(pa, 'PA Funded Projects')
pa.head(3)


  PA Funded Projects  |  810,656 rows  x  25 cols
Columns with nulls:
  applicantId                                5  (0.0%)
  county                                13,620  (1.7%)



,disasterNumber,declarationDate,incidentType,pwNumber,applicationTitle,applicantId,damageCategoryCode,damageCategoryDescrip,projectStatus,projectProcessStep,...,projectAmount,federalShareObligated,totalObligated,lastObligationDate,firstObligationDate,mitigationAmount,gmProjectId,gmApplicantId,lastRefresh,hash
0,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),1,(PW# 1) IMMEDIATE NEEDS FUNDING,465-19792-00,B,Emergency Protective Measures,Active,Project Closed Out,...,100000.0,75000.00,80340.00,1998-09-15T14:25:07.000Z,1998-09-15T14:25:07.000Z,0.0,1021769,268458,2025-11-27T15:05:59.253Z,addcfded82ae348f46ff034a4564f983a9dea897
1,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),5,(PW# 5) Not Provided,465-19792-00,G,"Parks, Recreational Facilities, and Other Items",Active,Project Closed Out,...,19685.5,14764.13,15461.00,1998-09-23T08:58:52.000Z,1998-09-23T08:58:52.000Z,0.0,1062596,268458,2025-11-27T15:05:59.253Z,05c6c522b930a9c38b52a0c4b0de853e98b4cb75
2,1239,1998-08-26T00:00:00.000Z,Severe Storm(s),7,(PW# 7) Not Provided,465-19792-00,G,"Parks, Recreational Facilities, and Other Items",Active,Project Closed Out,...,26111.0,19583.25,20507.58,1998-09-23T08:58:52.000Z,1998-09-23T08:58:52.000Z,0.0,1062598,268458,2025-11-27T15:05:59.253Z,0addbfed02721821348612482bfe36d8fe587d0f


## 1.2 Load FEMA Disaster Declarations

The disaster declarations file (69,766 rows) contains one row per state-disaster combination, meaning a single Presidential declaration generates multiple rows if it covers multiple states. Two filtering steps are applied here.

First, only disasters where `paProgramDeclared == 1` are retained, as the project is scoped to PA funding specifically. Second, the dataset is deduplicated to one row per `disasterNumber`, keeping only the incident begin and end dates. All other declaration columns are already present in the PA project data or are not needed for modeling.

The incident dates are the most important addition from this file. They define the time window of each disaster, which is used later to engineer `declaration_lag_days` and `incident_duration_days` — both of which serve as proxies for how quickly the federal government responded and how prolonged the event was.

## 1.2 Load FEMA Disaster Declarations

Similarly to the project one, the disaster declarations dataset contains one row per state-disaster combo. This means a single Presidential declaration generates multiple rows if it covers multiple states.
Here is the link to access it:
- https://www.fema.gov/openfema-data-page/disaster-declarations-summaries-v2

We applied two filtering steps here:
1. First, we only keep disasters where "paProgramDeclared == 1", as this project focuses strictly on PA funding.
2. Also we deduplicate the dataset to one row per "disasterNumber", keeping only the incident begin and end dates.

The incident dates are our most important addition from this file. They define the time window of each disaster, which we use later to engineer the "declaration_lag_days" and "incident_duration_days" features. These two serve as proxies for how quickly the federal government responded to the event and to see how prolonged the event was.

In [3]:
#loading full declarations file
decl = pd.read_csv(RAW + 'disaster_declarations.csv', low_memory=False)
data_summary(decl, 'Disaster Declarations')

#filter to pa-declared only -> drops non-pa disaster types
decl_pa = decl[decl['paProgramDeclared'] == 1].copy()
#deduplicate to one row per disaster + keep only date columns needed
decl_unique = (
    decl_pa
    .sort_values('incidentBeginDate')
    .drop_duplicates(subset='disasterNumber', keep='first')
    [['disasterNumber', 'incidentBeginDate', 'incidentEndDate']]
)
print(f'Unique PA disasters: {len(decl_unique):,}')
decl_unique.head(3)


  Disaster Declarations  |  69,766 rows  x  28 cols
Columns with nulls:
  incidentEndDate                          534  (0.8%)
  disasterCloseoutDate                  16,505  (23.7%)
  lastIAFilingDate                      50,333  (72.1%)
  designatedIncidentTypes               47,812  (68.5%)

Unique PA disasters: 4,951


,disasterNumber,incidentBeginDate,incidentEndDate
69765,1,1953-05-02T00:00:00.000Z,1953-05-02T00:00:00.000Z
69764,2,1953-05-15T00:00:00.000Z,1953-05-15T00:00:00.000Z
51472,3,1953-05-29T00:00:00.000Z,1953-05-29T00:00:00.000Z


## 1.3 Merge PA Projects with Declarations and Build FIPS

Here we join the PA project data to the declarations file using "disasterNumber" to attach incident dates to each project row. We use a left join to ensure no project records are dropped.

After the merge, we constructed a 5-digit Federal Information Processing Standards (FIPS) code by concatenating the state code and the county code. This is needed as later we attach the Census and NRI data and without a consistent geographic key, evaluating the localized socioeconomic impact of these disasters would be impossible.

In [4]:
#left join to keep all project rows + attach incident dates
merged = pa.merge(decl_unique, on='disasterNumber', how='left')
print(f'After merge: {merged.shape}')

#build 5-digit fips: state (2 digits) + county (3 digits)
#zfill ensures leading zeros are preserved -> e.g. 01001 not 001 (!! v important)
merged['fips'] = (
    merged['stateNumberCode'].astype(str).str.zfill(2) +
    merged['countyCode'].astype(str).str.zfill(3)
)
print('Sample FIPS values:', merged['fips'].head().tolist())

After merge: (810656, 27)
Sample FIPS values: ['48465', '48465', '48465', '22075', '48465']


## 1.4 to 1.6 — Supplementary County-Level Datasets

Three additional datasets are joined to enrich each disaster record with the socioeconomic and risk profile of the affected counties. All three are publicly available federal datasets and join to the project data using the FIPS code constructed in section 1.3.

**Census Median Household Income (ACS S1903):** Income level captures the fiscal capacity of the affected area. Lower-income counties tend to have less self-funding ability, which may influence how much of the recovery cost falls on federal PA grants.

**Census Poverty Rate (ACS S1701):** Poverty rate is a direct measure of community vulnerability. Counties with higher poverty rates have fewer private resources to draw on before and after a disaster, making them more dependent on federal assistance.

**FEMA National Risk Index (NRI):** The NRI is FEMA's own composite score summarising each county's exposure to 18 natural hazard types, weighted by expected annual loss and community resilience. Including it aligns the model with FEMA's internal risk framework and provides a single number that captures long-run hazard exposure without requiring the model to handle all 18 hazard dimensions separately. Population is also sourced from this file as it is already county-matched.

## 1.4 to 1.6 Supplementary County-Level Datasets

At this stage, we join 3 additional datasets to enrich each disaster record with the socioeconomic and risk profile of the affected counties. All three are publicly available federal datasets, for which we share the links to access them (for last 2 we applied filtering based on our projects needs so check those when downloading):
- https://www.fema.gov/flood-maps/products-tools/national-risk-index
- https://data.census.gov/table?q=median+income
- https://data.census.gov/table?q=s1701

 We chose to incorporate these as they align with our business objective of measuring the true financial vulnerability of a community beyond admin data.

**Census Median Household Income:** income level captures the fiscal capacity of the affected area. Lower-income counties tend to have less self-funding ability, which directly influences how much of the recovery cost falls on federal PA grants.

**Census Poverty Rate:** poverty rate is a direct measure of community vulnerability. Counties with higher poverty rates have fewer private resources to draw on before and after a disaster, making them highly dependent on federal assistance.

**FEMA National Risk Index (NRI):** the NRI is FEMA's internal composite score summarizing a county's exposure to 18 natural hazard types, weighted by expected annual loss and community resilience. Including this provides a single number that captures long-run hazard exposure, simplifying the model architecture while aligning with FEMA's actual risk framework.

## 1.4 Load Census Income (ACS S1903)

In [5]:
#skiprows=[1] drops the human-readable label row -> column codes become headers
income_raw = pd.read_csv(RAW + 'median_household_income.csv', skiprows=[1], low_memory=False)
print(f'Income shape: {income_raw.shape}')
#sanity check: national file has ~1 row, county file has ~3200
assert income_raw.shape[0] > 100, 'ERROR: looks like national totals — re-download at county level'

#keep geo_id + target column only -> drop all other acs estimates
income = income_raw[['GEO_ID', 'S1903_C03_001E']].copy()
income.columns = ['geo_id', 'median_income']
#extract last 5 chars of geo_id -> fips code
income['fips']         = income['geo_id'].astype(str).str[-5:]
#coerce to numeric -> non-numeric census codes (e.g. '-') become nan
income['median_income'] = pd.to_numeric(income['median_income'], errors='coerce')
income = income[['fips', 'median_income']].dropna()
print(f'Income clean: {income.shape}  |  nulls: {income["median_income"].isna().sum()}')
income.head(3)

Income shape: (3220, 243)
Income clean: (3220, 2)  |  nulls: 0


,fips,median_income
0,01001,58731
1,01003,58320
2,01005,32525


## 1.5 Load Census Poverty Rate (ACS S1701)

In [6]:
#same loading pattern as income -> skiprows + extract fips from geo_id
pov_raw = pd.read_csv(RAW + 'poverty_rate.csv', skiprows=[1], low_memory=False)
print(f'Poverty shape: {pov_raw.shape}')
assert pov_raw.shape[0] > 100, 'ERROR: looks like national totals — re-download at county level'

pov = pov_raw[['GEO_ID', 'S1701_C03_001E']].copy()
pov.columns = ['geo_id', 'poverty_rate']
pov['fips']         = pov['geo_id'].astype(str).str[-5:]
pov['poverty_rate'] = pd.to_numeric(pov['poverty_rate'], errors='coerce')
pov = pov[['fips', 'poverty_rate']].dropna()
print(f'Poverty clean: {pov.shape}')
pov.head(3)

Poverty shape: (3220, 369)
Poverty clean: (3220, 2)


,fips,poverty_rate
0,01001,15.2
1,01003,10.4
2,01005,30.7


## 1.6 Load FEMA National Risk Index (NRI)

In [7]:
#nri has 465 columns -> keep only fips, population, risk_score, risk_rating
nri_raw = pd.read_csv(RAW + 'national_risk_index.csv', low_memory=False)
print(f'NRI shape: {nri_raw.shape}')

nri = nri_raw[['STCOFIPS', 'POPULATION', 'RISK_SCORE', 'RISK_RATNG']].copy()
nri.columns = ['fips', 'population', 'risk_score', 'risk_rating']
#zfill to match 5-digit format used elsewhere
nri['fips']       = nri['fips'].astype(str).str.zfill(5)
nri['population'] = pd.to_numeric(nri['population'], errors='coerce')
nri['risk_score'] = pd.to_numeric(nri['risk_score'], errors='coerce')
print(f'NRI clean: {nri.shape}')
nri.head(3)

NRI shape: (3232, 465)
NRI clean: (3232, 4)


,fips,population,risk_score,risk_rating
0,01001,58764,57.569975,Relatively Low
1,01003,231365,96.723919,Relatively High
2,01005,25160,48.123410,Relatively Low


## 1.7 Combine County Features and Merge into Project Data

We combine the 3 county-level files into a single lookup table using outer joins on FIPS, ensuring that counties present in only one source are still retained. I then join this lookup table to the full project dataset.

The result is that every project row now carries the socioeconomic profile of the county where the project occurred. A known data risk at this stage is the presence of null values, as not all FIPS codes in the FEMA data have an exact match in the Census files (e.g., US territories like Puerto Rico). These missing values are handled through imputation in Notebook 02 to prevent data loss.

In [8]:
#outer join across all 3 county files -> preserves all fips even if one source missing
county_features = income.merge(pov, on='fips', how='outer').merge(nri, on='fips', how='outer')
print(f'County features: {county_features.shape}')

#left join to project data on fips -> preserves all 810k project rows
merged = merged.merge(county_features, on='fips', how='left')
print(f'Final project-level shape: {merged.shape}')
data_summary(merged, 'Merged Project-Level')

County features: (3241, 6)
Final project-level shape: (810656, 33)

  Merged Project-Level  |  810,656 rows  x  33 cols
Columns with nulls:
  applicantId                                5  (0.0%)
  county                                13,620  (1.7%)
  median_income                         20,890  (2.6%)
  poverty_rate                          20,890  (2.6%)
  population                            26,043  (3.2%)
  risk_score                            49,701  (6.1%)
  risk_rating                           26,043  (3.2%)



## 1.8 Aggregate to Disaster Level

Our model operates at the disaster level, not the project level. A single declared disaster can generate hundreds of individual PA project work orders across multiple counties. This section collapses the 810,656 project rows into 1,766 disaster rows by grouping on "disasterNumber".

The aggregation choices reflect what information is actually knowable at the moment of declaration:

* "total_federal_share": The sum of all project obligations. This becomes the target variable after CPI adjustment and binning in Notebook 02.
* "n_counties": The count of distinct FIPS codes affected. This measures the geographic breadth of the disaster.
* "population": Summed across counties, capturing the total population in the affected area.
* "median_income", "poverty_rate", "risk_score": The median across counties, representing the typical socioeconomic profile of the disaster footprint.
* Incident type, state, and dates: Taken from the first project row, since these are constant within a disaster.

We wanted to highlight that "n_projects" is computed here for completeness but is deliberately excluded from the modeling features in Notebook 04. Project counts are not known at declaration time and would constitute data leakage.

In [9]:
#groupby disaster -> collapse 810k rows to 1766 rows
disaster_level = merged.groupby('disasterNumber').agg(
    total_federal_share = ('federalShareObligated', 'sum'),   #sum all project obligations -> prediction target
    n_projects          = ('federalShareObligated', 'count'), #count projects -> excluded from model (leakage)
    n_counties          = ('fips', 'nunique'),                #geographic breadth of disaster
    #disaster attributes are constant per disaster -> take first row
    incidentType        = ('incidentType', 'first'),
    stateCode           = ('stateNumberCode', 'first'),
    stateAbbreviation   = ('stateAbbreviation', 'first'),
    declarationDate     = ('declarationDate', 'first'),
    incidentBeginDate   = ('incidentBeginDate', 'first'),
    incidentEndDate     = ('incidentEndDate', 'first'),
    #sum population across counties -> total affected population
    population          = ('population', 'sum'),
    #median for rates + scores -> avoids large counties dominating
    median_income       = ('median_income', 'median'),
    poverty_rate        = ('poverty_rate', 'median'),
    risk_score          = ('risk_score', 'median'),
).reset_index()

print(f'Disaster-level shape: {disaster_level.shape}')
disaster_level.head(3)

Disaster-level shape: (1766, 14)


,disasterNumber,total_federal_share,n_projects,n_counties,incidentType,stateCode,stateAbbreviation,declarationDate,incidentBeginDate,incidentEndDate,population,median_income,poverty_rate,risk_score
0,1239,7950369.84,276,6,Severe Storm(s),48,TX,1998-08-26T00:00:00.000Z,1998-08-22T00:00:00.000Z,1998-08-31T00:00:00.000Z,48941150.0,46147.0,20.3,76.081425
1,1257,32229051.37,1760,38,Flood,48,TX,1998-10-21T00:00:00.000Z,1998-10-17T00:00:00.000Z,1998-11-15T00:00:00.000Z,788095965.0,57157.0,14.1,86.482188
2,1260,9485830.00,212,35,Severe Storm(s),47,TN,1999-01-15T00:00:00.000Z,1998-12-23T00:00:00.000Z,1998-12-29T00:00:00.000Z,8042506.0,49485.0,15.8,60.909669


## 1.9 Save Output

We save the disaster-level dataset to our processed data folder. This file is the direct input to Notebook 02, where cleaning, feature engineering, and label creation take place.

In [10]:
#save to processed folder -> input for nb02
disaster_level.to_csv(PROCESSED + 'merged_disaster_level.csv', index=False)
print(f'Saved: merged_disaster_level.csv  ->  {disaster_level.shape}')

Saved: merged_disaster_level.csv  ->  (1766, 14)
